In [ ]:
# Before Type Hints
def process_user(user):
  return user.name.upper()

# We dont know what is user

# With type hints
def process_user(user: dict) -> str:
  # Its clear user is a dict return a string
  return user['name'].upper()

In [ ]:
# Basic Type Hints
# Variables
name: str = "John"
age: int = 30
price: float = 19.99
is_active: bool = True

# Functions
def greet(name: str) -> str:
    return f"Hello, {name}!"

def add(a: int, b: int) -> int:
    return a + b

def no_return() -> None:
    print("This function returns nothing")

# Lists
numbers: list[int] = [1, 2, 3, 4, 5]
names: list[str] = ["John", "Jane", "Bob"]

# Dictionaries
user: dict[str, int] = {"age": 30, "score": 100}

# Tuples
coordinates: tuple[float, float] = (10.5, 20.3)

# Sets
tags: set[str] = {"python", "backend", "api"}

In [ ]:
# Complex Types
from typing import List, Dict, Tuple, Set, Optional, Union, Any

# Optional (can be value or None)
def find_user(user_id: int) -> Optional[dict]:
    """Returns user dict or None if not found."""
    if user_id == 1:
        return {"id": 1, "name": "John"}
    return None

# Union (can be multiple types)
def process_id(user_id: Union[int, str]) -> str:
    """Accept int or string ID."""
    return str(user_id)

# List with specific type
def get_scores() -> List[int]:
    return [95, 87, 92, 88]

# Dict with specific key/value types
def get_user_ages() -> Dict[str, int]:
    return {"john": 30, "jane": 25}

# Tuple with specific types
def get_coordinates() -> Tuple[float, float]:
    return (10.5, 20.3)

# Any (accepts any type - avoid when possible)
def process_data(data: Any) -> Any:
    return data

In [ ]:
# Type hints for classes
from typing import List, Optional

class User:
    """User class with type hints."""

    def __init__(self, name: str, age: int, email: Optional[str] = None):
        self.name: str = name
        self.age: int = age
        self.email: Optional[str] = email
        self.friends: List[User] = []

    def add_friend(self, friend: 'User') -> None:
        """Add a friend (note: use quotes for forward references)."""
        self.friends.append(friend)

    def get_friend_names(self) -> List[str]:
        """Get list of friend names."""
        return [friend.name for friend in self.friends]

# Usage
user1 = User("John", 30, "john@example.com")
user2 = User("Jane", 25)

user1.add_friend(user2)
print(user1.get_friend_names())

In [ ]:
# Generic Types
from typing import TypeVar, Generic, List

T = TypeVar('T')  # Generic type variable

class Stack(Generic[T]):
    """Generic stack implementation."""

    def __init__(self):
        self.items: List[T] = []

    def push(self, item: T) -> None:
        self.items.append(item)

    def pop(self) -> T:
        return self.items.pop()

    def is_empty(self) -> bool:
        return len(self.items) == 0

# Usage
int_stack: Stack[int] = Stack()
int_stack.push(1)
int_stack.push(2)
print(int_stack.pop())  # 2

str_stack: Stack[str] = Stack()
str_stack.push("hello")
str_stack.push("world")
print(str_stack.pop())  # world

In [ ]:
from typing import List, Optional
from pydantic import BaseModel
from fastapi import FastAPI

# Pydantic models (FastAPI uses these)
class User(BaseModel):
    """User model with validation."""
    id: int
    name: str
    email: str
    age: Optional[int] = None
    tags: List[str] = []

class CreateUserRequest(BaseModel):
    """Request body for creating user."""
    name: str
    email: str
    age: Optional[int] = None

app = FastAPI()

@app.post("/users", response_model=User)
def create_user(user: CreateUserRequest) -> User:
    """
    Create a new user.

    - FastAPI automatically validates the request body
    - Type hints define the API contract
    """
    # In reality, save to database
    new_user = User(
        id=1,
        name=user.name,
        email=user.email,
        age=user.age,
        tags=[]
    )
    return new_user

@app.get("/users/{user_id}", response_model=User)
def get_user(user_id: int) -> User:
    """
    Get user by ID.

    - FastAPI converts user_id to int automatically
    - Returns a User object
    """
    # In reality, fetch from database
    return User(id=user_id, name="John", email="john@example.com")

@app.get("/users", response_model=List[User])
def list_users(skip: int = 0, limit: int = 10) -> List[User]:
    """
    List users with pagination.

    - Query parameters have default values
    - Returns a list of User objects
    """
    # In reality, fetch from database
    return [
        User(id=1, name="John", email="john@example.com"),
        User(id=2, name="Jane", email="jane@example.com"),
    ]

In [ ]:
# Typing for RunTime Validation
from pydantic import BaseModel, validator

class User(BaseModel):
    """User with runtime validation."""
    name: str
    age: int
    email: str

    @validator('age')
    def age_must_be_positive(cls, v):
        if v < 0:
            raise ValueError('Age must be positive')
        return v

    @validator('email')
    def email_must_contain_at(cls, v):
        if '@' not in v:
            raise ValueError('Invalid email')
        return v

# This works
user1 = User(name="John", age=30, email="john@example.com")

# This raises ValidationError
try:
    user2 = User(name="Jane", age=-5, email="invalid")
except Exception as e:
    print(f"Validation error: {e}")